In [1]:
!pip install -q groq tiktoken
import os
import pandas as pd
from google.colab import userdata
from groq import Groq
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
client = Groq()
MODEL = "qwen/qwen3.8-27b" # same model as Lab 7, Playground A


In [2]:
def ask_llm(prompt, temperature=0.7, top_p=0.95, max_tokens=1024):
    """Send a prompt to the LLM. Returns (reply, usage, finish_reason)."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
        reasoning_effort="none",   # thinking off, same as Lab 7
    )
    reply = response.choices[0].message.content
    finish_reason = response.choices[0].finish_reason
    return reply, response.usage, finish_reason

In [3]:
reply, usage, finish = ask_llm("Recommend 3 gym earbuds under Rs. 2,000,one line each.")

print(reply)
print("\nInput tokens :", usage.prompt_tokens)
print("Output tokens:", usage.completion_tokens)
print("Total tokens :", usage.total_tokens)
print("Finish reason:", finish)

Here are 3 recommended gym earbuds under Rs. 2,000:

1. **JBL Wave Beam** – Offers reliable JBL sound and IPX5 water resistance for sweat-proof workouts.
2. **boAt Airdopes 131** – Provides 42-hour battery life and secure fit at a very affordable price point.
3. **Noise Buds VS 4** – Features a comfortable ergonomic design with clear sound and stable Bluetooth 5.0 connectivity.

Input tokens : 33
Output tokens: 107
Total tokens : 140
Finish reason: stop


In [4]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")
def count_tokens(text):
    """Return the number of tokens in a piece of text."""
    return len(enc.encode(text))

def show_tokens(text):
    """Print word count, token count and every token piece."""
    pieces = [enc.decode([t]) for t in enc.encode(text)]
    print(f"{len(text.split())} words -> {len(pieces)} tokens")
    print(pieces, "\n")

show_tokens("Recommend 3 gym earbuds under Rs. 2,000")
show_tokens("office ke liye formal shirt chahiye under 1500 rupees")

7 words -> 14 tokens
['Recommend', ' ', '3', ' gym', ' ear', 'b', 'uds', ' under', ' Rs', '.', ' ', '2', ',', '000'] 

9 words -> 15 tokens
['office', ' ke', ' li', 'ye', ' formal', ' shirt', ' ch', 'ahi', 'ye', ' under', ' ', '150', '0', ' ru', 'pees'] 



In [6]:
prompts = pd.read_csv("/content/smartcart_llm_test_prompts.csv")

prompts["words"] = prompts["prompt"].str.split().str.len()
prompts["tokens"] = prompts["prompt"].apply(count_tokens)
prompts["tokens_per_word"] = (prompts["tokens"] / prompts["words"]).round(2)

print(prompts[["id", "type", "words", "tokens", "tokens_per_word"]])

   id                    type  words  tokens  tokens_per_word
0  P1  Product recommendation     10      19             1.90
1  P2              Long query     20      21             1.05
2  P3              Comparison     10      14             1.40
3  P4          Hinglish query      9      15             1.67
4  P5            Store policy      8      11             1.38
5  P6       Structured output     18      22             1.22


In [8]:
MODEL = "qwen/qwen3.8-27b"
CONTEXT_WINDOW = 32768
print(f"{MODEL} context window: {CONTEXT_WINDOW:,} tokens")

qwen/qwen3.8-27b context window: 32,768 tokens


In [9]:
def fits_in_context(prompt, max_tokens=1024):
    """Return True if the prompt plus the reply budget fits."""
    needed = count_tokens(prompt) + max_tokens
    print(f"Needed {needed:,} of {CONTEXT_WINDOW:,} tokens")
    return needed <= CONTEXT_WINDOW

queries = pd.read_csv("/content/smartcart_customer_queries.csv")
one_day_log = "\n".join(queries["query_text"].str.strip())
fifty_day_log = "\n".join([one_day_log] * 50)   # pretend 50 days of logs

print("1 day of queries  :", fits_in_context(one_day_log))
print("50 days of queries:", fits_in_context(fifty_day_log))

Needed 4,200 of 32,768 tokens
1 day of queries  : True
Needed 159,824 of 32,768 tokens
50 days of queries: False


In [10]:
prompt = "Suggest one name for a SmartCart Diwali grocery combo. Reply with the name only."

for temp in [0.0, 1.0, 1.8]:
    print(f"Temperature {temp}")
    for run in range(3):
        reply, usage, finish = ask_llm(prompt, temperature=temp, max_tokens=20)
        print("   ", reply.strip())

Temperature 0.0
    Diwali Dhamaka
    Diwali Dhamaka
    Diwali Dhamaka
Temperature 1.0
    Diwali Dhamaka
    Diwali Delight
    Diywali Delight
Temperature 1.8
    Diyar Jasher Bundel
    Divine Delights Pannieu
    SparkleStaples


In [11]:
for p in [0.1, 1.0]:
    print(f"Top P {p}")
    for run in range(3):
        reply, usage, finish = ask_llm(prompt, temperature=1.0, top_p=p, max_tokens=20)
        print("   ", reply.strip())

Top P 0.1
    Diwali Dhamaka
    Diwali Dhamaka
    Diwali Dhamaka
Top P 1.0
    Utsav Basket
    MangoDiya
    Luminous Harvest


In [12]:
prompt = "Write a short buying guide for choosing a pressure cooker."

for limit in [30, 150, 600]:
    reply, usage, finish = ask_llm(prompt, max_tokens=limit)
    print(f"max_tokens={limit} -> used {usage.completion_tokens}, finish_reason={finish}")
    print("   ends with: ...", reply.strip()[-60:], "\n")

max_tokens=30 -> used 30, finish_reason=length
   ends with: ... of the most versatile and time-saving appliances you can own 

max_tokens=150 -> used 150, finish_reason=length
   ends with: ... slow cooker, steamer, and yogurt maker), no need to monitor. 

max_tokens=600 -> used 600, finish_reason=length
   ends with: ... es, larger capacities, or high-end brands.

### Final Tips
* 



In [13]:
import os
from groq import Groq

MODEL = "qwen/qwen3.8-27b"
client = Groq(api_key=os.environ["GROQ_API_KEY"])


def recommend(request, temperature=0.7, top_p=0.95, max_tokens=300):
    """Return (reply, usage, finish_reason) for one shopper request."""
    prompt = (
        "You are SmartCart's shopping assistant. Recommend 3 products "
        "for the request below, one line each, with an approximate "
        f"price in rupees.\nRequest: {request}")

import os
from groq import Groq

MODEL = "qwen/qwen3.8-27b"
client = Groq(api_key=os.environ["GROQ_API_KEY"])


def recommend(request, temperature=0.7, top_p=0.95, max_tokens=300):
    """Return (reply, usage, finish_reason) for one shopper request."""
    prompt = (
        "You are SmartCart's shopping assistant. Recommend 3 products "
        "for the request below, one line each, with an approximate "
        f"price in rupees.\nRequest: {request}")

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
        reasoning_effort="none",)

    choice = response.choices[0]
    return choice.message.content, response.usage, choice.finish_reason


def get_setting(name, default, low, high, number_type):
    """Ask for a setting. Blank, invalid or out-of-range input uses the default."""
    text = input(f"{name} ({low} to {high}, Enter = {default}): ").strip()
    if text == "":
        return default
    try:
        value = number_type(text)
    except ValueError:
        print(f"   Not a number, using {default}")
        return default
    if low <= value <= high:
        return value
    print(f"   Out of range, using {default}")
    return default

import os
from groq import Groq

MODEL = "qwen/qwen3.8-27b"
client = Groq(api_key=os.environ["GROQ_API_KEY"])


def recommend(request, temperature=0.7, top_p=0.95, max_tokens=300):
    """Return (reply, usage, finish_reason) for one shopper request."""
    prompt = (
        "You are SmartCart's shopping assistant. Recommend 3 products "
        "for the request below, one line each, with an approximate "
        f"price in rupees.\nRequest: {request}")

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
        reasoning_effort="none",)

    choice = response.choices[0]
    return choice.message.content, response.usage, choice.finish_reason


def get_setting(name, default, low, high, number_type):
    """Ask for a setting. Blank, invalid or out-of-range input uses the default."""
    text = input(f"{name} ({low} to {high}, Enter = {default}): ").strip()
    if text == "":
        return default
    try:
        value = number_type(text)
    except ValueError:
        print(f"   Not a number, using {default}")
        return default
    if low <= value <= high:
        return value
    print(f"   Out of range, using {default}")
    return default

if __name__ == "__main__":
    request = input("What are you shopping for? ")
    temperature = get_setting("Temperature", 0.7, 0.0, 2.0, float)
    top_p = get_setting("Top P", 0.95, 0.0, 1.0, float)
    max_tokens = get_setting("Max tokens", 300, 1, 2048, int)

    reply, usage, finish = recommend(request, temperature, top_p, max_tokens)

    print("\n" + reply)
    print(f"\n[tokens in: {usage.prompt_tokens} | out: {usage.completion_tokens}"
          f" | finish: {finish}]")
    if finish == "length":
        print("Reply was cut off -- try a higher Max tokens.")

What are you shopping for? running shoes for women under 3000
Temperature (0.0 to 2.0, Enter = 0.7): 0.7
Top P (0.0 to 1.0, Enter = 0.95): 0.95
Max tokens (1 to 2048, Enter = 300): 300

1. ASICS Gel-Contend 7 Running Shoes for Women – Approx. ₹2,499
2. Nike Revolution 6 Women's Road Running Shoes – Approx. ₹2,899
3. Puma Cell V3 RnR Running Shoes for Women – Approx. ₹2,199

[tokens in: 54 | out: 69 | finish: stop]


In [14]:
presets = [
    {"preset": "Safe",     "temperature": 0.2, "top_p": 0.9,  "max_tokens": 200},
    {"preset": "Balanced", "temperature": 0.7, "top_p": 0.95, "max_tokens": 300},
    {"preset": "Creative", "temperature": 1.3, "top_p": 1.0,  "max_tokens": 300},]
request = "healthy snacks for office under 500 rupees"

rows = []
for s in presets:
    reply, usage, finish = recommend(request, s["temperature"], s["top_p"], s["max_tokens"])
    rows.append({**s, "output_tokens": usage.completion_tokens,
                 "finish_reason": finish, "reply": reply})

log = pd.DataFrame(rows)
log.to_csv("smartcart_lab8_settings_log.csv", index=False)
print(log[["preset", "temperature", "top_p", "max_tokens",
           "output_tokens", "finish_reason"]])

     preset  temperature  top_p  max_tokens  output_tokens finish_reason
0      Safe          0.2   0.90         200             62          stop
1  Balanced          0.7   0.95         300             62          stop
2  Creative          1.3   1.00         300             72          stop


Lab no 8


In [15]:
!pip install -q langchain langchain-groq
from importlib.metadata import version
print("langchain      :", version("langchain"))
print("langchain-groq :", version("langchain-groq"))

langchain      : 1.4.8
langchain-groq : 1.2.0


In [16]:
%%writefile requirements.txt
langchain==1.0.3
langchain-groq==1.0.1

Writing requirements.txt


In [17]:
%%writefile .gitignore
.env
__pycache__/

Writing .gitignore


In [18]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

key = os.environ["GROQ_API_KEY"]
print("Key loaded:", key[:4] + "..." + key[-4:])   # never print the full key
print("Key length:", len(key))

Key loaded: gsk_...7y88
Key length: 56


In [19]:
from langchain_groq import ChatGroq
bad_llm = ChatGroq(model="qwen/qwen3.8-27b", api_key="gsk_fake_key_123")
try:
    bad_llm.invoke("hello")
except Exception as error:
    print(type(error).__name__, "-", error)

AuthenticationError - Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}


In [20]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0.7,
    max_tokens=300,
    model_kwargs={"top_p": 0.95},   # top_p is passed through model_kwargs
    reasoning_effort="none",        # thinking off, same as Labs 7 and 8
)

response = llm.invoke("Recommend 3 gym earbuds under Rs. 2,000, one line each.")
print(response.content)

1. **boAt Airdopes 311 Pro** – Offers excellent value with deep bass and a secure fit for intense workouts.
2. **JBL Tune 215BT** – Provides reliable sound quality and a comfortable, ergonomic design for long gym sessions.
3. **Sony WF-C500** – Delivers balanced audio and durable build quality, ideal for consistent use in any fitness routine.


In [21]:
from langchain_core.messages import HumanMessage

request = [HumanMessage(content="Recommend 3 gym earbuds under Rs. 2,000, one line each.")]

for message in request:
    print("role   :", message.type)
    print("content:", message.content)

response = llm.invoke(request)

role   : human
content: Recommend 3 gym earbuds under Rs. 2,000, one line each.


In [22]:
print("Type          :", type(response).__name__)
print("Content       :", response.content[:60], "...")
print("Token usage   :", response.usage_metadata)
print("Model name    :", response.response_metadata["model_name"])
print("Finish reason :", response.response_metadata["finish_reason"])

Type          : AIMessage
Content       : 1. **JBL Tune 520BT**: Features strong bass and a secure fit ...
Token usage   : {'input_tokens': 33, 'output_tokens': 107, 'total_tokens': 140}
Model name    : qwen/qwen3.8-27b
Finish reason : stop


In [23]:
%%writefile smartcart_v1.py
import os
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

 # 1. Authentication: stop early with a clear message if the key is missing
if not os.environ.get("GROQ_API_KEY"):
    raise SystemExit("GROQ_API_KEY not found. Add it to Colab Secrets (Lab 9, Task 2).")

# 2. Model: created once, reused for every request
llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0.7,
    max_tokens=300,
    model_kwargs={"top_p": 0.95},
    reasoning_effort="none", )

def ask_smartcart(shopper_request):
    """Send one shopping request and return the AIMessage."""
    prompt = (
        "You are SmartCart's shopping assistant. Recommend 3 products "
        "for the request below, one line each, with an approximate "
        f"price in rupees.\nRequest: {shopper_request}" )
    request = [HumanMessage(content=prompt)]   # 3. Request structure
    return llm.invoke(request)                 # 4. Single API call

if __name__ == "__main__":
    shopper_request = input("What are you shopping for? ")
    response = ask_smartcart(shopper_request)

    usage = response.usage_metadata
    print("\n" + response.content)
    print(f"\n[model: {response.response_metadata['model_name']}"
          f" | tokens in: {usage['input_tokens']}"
          f" | out: {usage['output_tokens']}"
          f" | finish: {response.response_metadata['finish_reason']}]")

Writing smartcart_v1.py


In [24]:
%run smartcart_v1.py

What are you shopping for? office ke liye formal shirt chahiya under 1500 rupees

1. Van Heusen Men's Slim Fit Formal Shirt (White) - ₹1,299
2. Allen Solly Men's Classic Fit Formal Shirt (Light Blue) - ₹1,349
3. Peter England Men's Regular Fit Formal Shirt (Black) - ₹1,449

[model: qwen/qwen3.8-27b | tokens in: 61 | out: 66 | finish: stop]


In [25]:
from smartcart_v1 import ask_smartcart

tests = [
    "gym earbuds under Rs. 2,000",
    "smartphone with good camera and fast charging under 5000 rupees",
    "healthy snacks for office under 500 rupees", ]

for shopper_request in tests:
    response = ask_smartcart(shopper_request)
    usage = response.usage_metadata
    print(f"{shopper_request[:40]:<40} | out tokens: {usage['output_tokens']:>3}"
          f" | finish: {response.response_metadata['finish_reason']}")

gym earbuds under Rs. 2,000              | out tokens:  90 | finish: stop
smartphone with good camera and fast cha | out tokens: 157 | finish: stop
healthy snacks for office under 500 rupe | out tokens:  68 | finish: stop
